# Lọc nội dung · A · tín hiệu hình từng giây của 30 video

Mỗi giây: vector SigLIP base, điểm quảng cáo của model trò, điểm bộ gác OCR, độ đổi hình so với
giây trước. Lưu `signals/<video>.npz` để chỉnh cách **tự so với số đông** trên máy, không cần GPU.


In [ ]:
%pip install -q "av>=14,<17" "opencv-python-headless>=4.10,<4.12" "Pillow>=10" "huggingface-hub>=0.25,<1" "transformers==4.57.1"


In [ ]:
"""Shared artifact contract. Embedded into every notebook by build_notebooks.py."""

from pathlib import Path
import hashlib
import importlib.metadata
import json
import math
import re
import shutil
import subprocess
import threading
import unicodedata
from datetime import datetime, timezone

# transformers' from_pretrained builds models under a process-wide "meta device" context
# that is not thread-safe: two loads at once (OCR thread and main thread) fail with
# "Cannot copy out of meta tensor". Every model is loaded once per session, one at a time.
MODEL_LOAD_LOCK = threading.RLock()
_SHARED_MODELS = {}


def shared_model(key, loader):
    """Load a model once per session (reused by every video) and never two at once."""
    with MODEL_LOAD_LOCK:
        if key not in _SHARED_MODELS:
            _SHARED_MODELS[key] = loader()
        return _SHARED_MODELS[key]


SCHEMA_VERSION = "lecture-rag-v1"
PIPELINE_VERSION = "l25-v1"
STAGE_NAMES = {
    1: "01_slides", 2: "02_visual", 3: "03_audio",
    4: "04_bundle", 5: "05_index", 6: "06_retrieval",
}


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False),
        encoding="utf-8",
    )
    temporary.replace(path)


def fingerprint(value):
    data = json.dumps(value, ensure_ascii=False, sort_keys=True, allow_nan=False)
    return hashlib.sha256(data.encode("utf-8")).hexdigest()


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def relative_asset(root, name):
    root = Path(root).resolve()
    path = (root / name).resolve()
    if not path.is_relative_to(root):
        raise ValueError(f"Asset nằm ngoài thư mục kết quả: {name}")
    return path


def run_command(arguments):
    process = subprocess.run(
        [str(value) for value in arguments], capture_output=True, text=True,
        encoding="utf-8", errors="replace",
    )
    if process.returncode:
        raise RuntimeError(
            f"{arguments[0]} thất bại ({process.returncode}):\n{process.stderr[-3000:]}"
        )
    return process.stdout


def package_versions(names):
    versions = {}
    for name in names:
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            versions[name] = "not-installed"
    return versions


def find_video(override=None, input_root="/kaggle/input"):
    if override:
        path = Path(override)
        if not path.is_file():
            raise FileNotFoundError(f"Không tìm thấy video: {path}")
        return path
    candidates = sorted(
        path for path in Path(input_root).rglob("*")
        if path.is_file()
        and path.suffix.lower() in {".mp4", ".mkv", ".mov", ".webm"}
        and path.stem.lower() in {"l25_v001", "tl25_v001"}
    )
    if len(candidates) != 1:
        raise ValueError(
            "Cần đúng một video L25_V001/TL25_V001. Add dataset "
            "thnhtrungnguynmif/l25a-lecture-videos-media hoặc đặt VIDEO_PATH. "
            f"Tìm thấy: {[str(path) for path in candidates]}"
        )
    return candidates[0]


def probe_video(path):
    if not shutil.which("ffprobe") or not shutil.which("ffmpeg"):
        raise RuntimeError("Cần FFmpeg và FFprobe trong PATH.")
    data = json.loads(run_command([
        "ffprobe", "-v", "error", "-show_format", "-show_streams",
        "-of", "json", str(path),
    ]))
    videos = [stream for stream in data["streams"] if stream["codec_type"] == "video"]
    audios = [stream for stream in data["streams"] if stream["codec_type"] == "audio"]
    if not videos:
        raise ValueError("Tệp không có luồng video.")
    info = data.get("format", {})
    origin = float(info.get("start_time", 0) or 0)
    duration = float(info.get("duration") or videos[0].get("duration") or 0)
    if not math.isfinite(duration) or duration <= 0:
        raise ValueError("Không đọc được thời lượng video hợp lệ.")
    return {
        "filename": Path(path).name,
        "sha256": file_sha256(path),
        "duration_sec": duration,
        "timeline_origin_sec": origin,
        "video_start_sec": float(videos[0].get("start_time", origin)),
        "audio_start_sec": float(audios[0].get("start_time", origin)) if audios else None,
        "has_audio": bool(audios),
        "width": int(videos[0]["width"]),
        "height": int(videos[0]["height"]),
        "clock": "original_presentation_timestamp_minus_format_start_time",
    }


def begin_stage(output_root, stage):
    directory = Path(output_root) / STAGE_NAMES[stage]
    directory.mkdir(parents=True, exist_ok=True)
    write_json(directory / "_RUNNING.json", {
        "stage": stage, "started_utc": datetime.now(timezone.utc).isoformat(),
    })
    return directory


def finish_stage(directory, stage, video_id, source, parameters, parents=(), extra=None):
    directory = Path(directory)
    files = {
        path.relative_to(directory).as_posix(): file_sha256(path)
        for path in sorted(directory.rglob("*"))
        if path.is_file()
        and path.name != "manifest.json"
        and not any(part.startswith("_") for part in path.relative_to(directory).parts)
        and path.suffix != ".tmp"
    }
    payload = {
        "schema_version": SCHEMA_VERSION,
        "pipeline_version": PIPELINE_VERSION,
        "stage": stage,
        "video_id": video_id,
        "source": source,
        "parameters": parameters,
        "parents": {str(parent["stage"]): parent["artifact_id"] for parent in parents},
        "files": files,
        **(extra or {}),
    }
    payload["artifact_id"] = fingerprint(payload)
    payload["created_utc"] = datetime.now(timezone.utc).isoformat()
    write_json(directory / "manifest.json", payload)
    (directory / "_RUNNING.json").unlink(missing_ok=True)
    print(f"Hoàn tất {STAGE_NAMES[stage]}: {directory}")
    return payload


def load_stage(directory, stage=None, video_id="L25_V001", verify=True):
    directory = Path(directory)
    if (directory / "_RUNNING.json").exists():
        raise ValueError(f"Bước xử lý chưa hoàn tất; cần chạy lại: {directory}")
    manifest = read_json(directory / "manifest.json")
    if manifest.get("schema_version") != SCHEMA_VERSION:
        raise ValueError(f"Schema không tương thích: {directory}")
    if stage is not None and manifest.get("stage") != stage:
        raise ValueError(f"Cần stage {stage}, nhận {manifest.get('stage')}: {directory}")
    if manifest.get("video_id") != video_id:
        raise ValueError(f"Sai video_id: {manifest.get('video_id')} != {video_id}")
    if manifest.get("pipeline_version") != PIPELINE_VERSION:
        raise ValueError(f"Sai pipeline_version: {directory}")
    signed = {key: value for key, value in manifest.items()
              if key not in {"artifact_id", "created_utc"}}
    if fingerprint(signed) != manifest["artifact_id"]:
        raise ValueError(f"Manifest đã thay đổi sau khi xuất: {directory}")
    if verify:
        for name, expected in manifest["files"].items():
            path = relative_asset(directory, name)
            if not path.is_file() or file_sha256(path) != expected:
                raise ValueError(f"Asset bị thiếu hoặc không khớp checksum: {path}")
    return manifest


def find_stage(stage, override=None, video_id="L25_V001",
               search_roots=("/kaggle/input", "/kaggle/working")):
    if override:
        directory = Path(override)
        return directory, load_stage(directory, stage, video_id)
    matches = []
    for root in map(Path, search_roots):
        if not root.exists():
            continue
        for path in root.rglob("manifest.json"):
            try:
                manifest = read_json(path)
            except (ValueError, OSError):
                continue
            if (manifest.get("schema_version") == SCHEMA_VERSION
                    and manifest.get("stage") == stage
                    and manifest.get("video_id") == video_id
                    and manifest.get("pipeline_version") == PIPELINE_VERSION):
                matches.append((path.parent, manifest))
    identities = {manifest.get("artifact_id") for _, manifest in matches}
    if not matches or len(identities) != 1:
        raise ValueError(
            f"Cần một phiên bản rõ ràng của {STAGE_NAMES[stage]}. "
            "Add Input → Notebook Output của bước trước, hoặc đặt STAGE_XX_DIR. "
            f"Các thư mục tìm được: {[str(path) for path, _ in matches]}"
        )
    directory = sorted(matches, key=lambda item: str(item[0]))[0][0]
    return directory, load_stage(directory, stage, video_id)


def require_same_source(*manifests):
    identities = {
        (item["video_id"], item["source"]["sha256"], item["pipeline_version"])
        for item in manifests
    }
    if len(identities) != 1:
        raise ValueError("Các nhánh không được tạo từ cùng video/pipeline.")


def require_parent(child, parent):
    require_same_source(child, parent)
    if child["parents"].get(str(parent["stage"])) != parent["artifact_id"]:
        raise ValueError(
            f"Stage {child['stage']} dùng phiên bản stage {parent['stage']} khác. "
            "Chạy lại nhánh với cùng kết quả phát hiện slide."
        )


def verify_intervals(intervals, duration, tolerance=0.03):
    if not intervals:
        raise ValueError("Danh sách interval rỗng.")
    ids = set()
    previous_end = 0.0
    for index, item in enumerate(intervals):
        start, end = float(item["start_sec"]), float(item["end_sec"])
        if not (math.isfinite(start) and math.isfinite(end)
                and 0 <= start < end <= duration + tolerance):
            raise ValueError(f"Interval không hợp lệ: {item}")
        if abs(start - previous_end) > tolerance:
            raise ValueError(f"Khoảng trống/chồng lấn ở interval {index}.")
        if item["segment_id"] in ids:
            raise ValueError("Trùng segment_id.")
        ids.add(item["segment_id"])
        previous_end = end
    if abs(previous_end - duration) > tolerance:
        raise ValueError("Interval cuối không bao phủ hết video.")


def owner_interval(intervals, time_sec):
    # Half-open intervals: a word exactly at a boundary belongs to the next one.
    for item in intervals:
        if item["start_sec"] <= time_sec < item["end_sec"]:
            return item
    if intervals and math.isclose(time_sec, intervals[-1]["end_sec"], abs_tol=1e-6):
        return intervals[-1]
    raise ValueError(f"Timestamp nằm ngoài lecture: {time_sec}")


def copy_stage_assets(source, destination, manifest):
    for name in manifest["files"]:
        target = relative_asset(destination, name)
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(relative_asset(source, name), target)


def normalize_search(text):
    text = unicodedata.normalize("NFD", text.casefold().replace("đ", "d"))
    return "".join(char for char in text if not unicodedata.combining(char))


def timestamp_label(seconds):
    seconds = max(0, int(seconds))
    hours, seconds = divmod(seconds, 3600)
    minutes, seconds = divmod(seconds, 60)
    return f"{hours:02}:{minutes:02}:{seconds:02}" if hours else f"{minutes:02}:{seconds:02}"


def unique_records(records, key):
    result = {}
    for record in records:
        if record[key] in result:
            raise ValueError(f"Trùng {key}: {record[key]}")
        result[record[key]] = record
    return result


def read_pcm_audio(path, sample_rate=16000):
    import numpy as np
    import wave

    with wave.open(str(path), "rb") as stream:
        if (stream.getframerate() != sample_rate or stream.getnchannels() != 1
                or stream.getsampwidth() != 2):
            raise ValueError("Audio phải là PCM signed 16-bit, mono, 16 kHz.")
        return np.frombuffer(stream.readframes(stream.getnframes()), dtype="<i2").copy()


def save_pcm_clip(path, pcm, start_sample, end_sample, sample_rate=16000):
    import wave

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with wave.open(str(path), "wb") as stream:
        stream.setnchannels(1)
        stream.setsampwidth(2)
        stream.setframerate(sample_rate)
        stream.writeframes(pcm[start_sample:end_sample].astype("<i2").tobytes())


In [ ]:
"""PTS-based visual intervals, stable-change detection, and representative frames."""


def frame_signature(image, roi=(0, 0, 1, 1), ignored_regions=()):
    import numpy as np

    return frame_signature_array(np.asarray(image.convert("RGB")), roi, ignored_regions)


def frame_signature_array(pixels, roi=(0, 0, 1, 1), ignored_regions=()):
    import cv2
    import numpy as np

    pixels = pixels.copy()
    height, width = pixels.shape[:2]
    for left, top, right, bottom in ignored_regions:
        pixels[int(top * height):int(bottom * height),
               int(left * width):int(right * width)] = 127
    left, top, right, bottom = roi
    pixels = pixels[int(top * height):max(int(bottom * height), 1),
                    int(left * width):max(int(right * width), 1)]
    if pixels.size == 0:
        raise ValueError("SLIDE_ROI không hợp lệ.")
    gray = cv2.cvtColor(pixels, cv2.COLOR_RGB2GRAY)
    gray = cv2.resize(gray, (240, 135), interpolation=cv2.INTER_AREA)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    edges = cv2.GaussianBlur(cv2.Canny(gray, 50, 150), (3, 3), 0)
    # Text/layout edges catch two white slides whose average colors barely differ.
    return np.stack([blurred, edges]).astype("float32") / 255.0


def signature_distance(left, right):
    import numpy as np
    # Ignore the most changed 3% of pixels to reduce isolated cursor/overlay noise.
    differences = np.abs(left[0] - right[0]).reshape(-1)
    cutoff = max(1, int(len(differences) * 0.97))
    color_change = float(np.partition(differences, cutoff - 1)[:cutoff].mean())
    edge_change = float(np.abs(left[1] - right[1]).mean())
    edge_density = max(0.08, float(left[1].mean() + right[1].mean()))
    # Either a palette change or a text/layout change is sufficient. Averaging
    # them dilutes text-only changes on otherwise identical white backgrounds.
    return min(1.0, max(color_change, edge_change / edge_density))


def sampled_frames(video_path, source, sample_hz=1.0, roi=(0, 0, 1, 1),
                   ignored_regions=()):
    import av

    next_time = 0.0
    with av.open(str(video_path)) as container:
        stream = container.streams.video[0]
        stream.thread_type = "AUTO"
        for frame in container.decode(stream):
            if frame.pts is None:
                continue
            timestamp = float(frame.pts * frame.time_base) - source["timeline_origin_sec"]
            if timestamp < 0 or timestamp + 1e-6 < next_time:
                continue
            if timestamp >= source["duration_sec"]:
                break
            yield {
                "time_sec": timestamp,
                "signature": frame_signature(frame.to_image(), roi, ignored_regions),
            }
            next_time = timestamp + 1.0 / sample_hz


def scan_frames(video_path, source, directory, sample_hz=1.0, fine_hz=8.0,
                roi=(0, 0, 1, 1), ignored_regions=(), keep_width=1280,
                skip_nonref=True, records=None):
    """Decode once. Yields 1/sample_hz samples; each carries the fine-rate
    signatures since the previous sample (for boundary refinement without
    seeking) and a saved frame for keyframe selection."""
    import av
    import cv2

    width, height = source["width"], source["height"]
    small_width = min(480, width)
    small_height = max(2, round(height * small_width / width / 2) * 2)
    keep_width = min(keep_width, width)
    keep_height = max(2, round(height * keep_width / width / 2) * 2)
    step, fine_step = 1.0 / sample_hz, 1.0 / max(fine_hz, sample_hz)
    (Path(directory) / "_scan").mkdir(parents=True, exist_ok=True)
    next_sample = next_fine = 0.0
    fine, index = [], 0
    with av.open(str(video_path)) as container:
        stream = container.streams.video[0]
        stream.thread_type = "AUTO"
        if skip_nonref:
            # Non-reference frames are never needed at 1-4 Hz; skipping them
            # roughly halves H.264 decoding with B-frames. PTS stay exact.
            stream.codec_context.skip_frame = "NONREF"
        for frame in container.decode(stream):
            if frame.pts is None:
                continue
            timestamp = float(frame.pts * frame.time_base) - source["timeline_origin_sec"]
            if timestamp < 0 or timestamp + 1e-6 < next_fine:
                continue
            if timestamp >= source["duration_sec"]:
                break
            pixels = frame.to_ndarray(format="rgb24", width=small_width, height=small_height)
            signature = frame_signature_array(pixels, roi, ignored_regions)
            next_fine = timestamp + fine_step
            if timestamp + 1e-6 < next_sample:
                fine.append((timestamp, signature))
                continue
            gray = cv2.cvtColor(pixels, cv2.COLOR_RGB2GRAY)
            name = f"_scan/sample_{index:06d}.jpg"
            frame.to_image(width=keep_width, height=keep_height).save(
                Path(directory) / name, quality=90)
            sample = {"time_sec": timestamp, "signature": signature, "fine": fine,
                      "sharpness": float(cv2.Laplacian(gray, cv2.CV_64F).var()),
                      "image_path": name}
            if records is not None:
                record = {key: sample[key] for key in ("time_sec", "sharpness", "image_path")}
                # uint8 keeps a 30-minute video's signatures near 120 MB.
                record["signature"] = (signature * 255).round().astype("uint8")
                records.append(record)
            yield sample
            fine, index = [], index + 1
            next_sample = timestamp + step


def motion_mask(records, pixel_change=0.10, busy_fraction=0.30):
    """Pixels that change in most seconds of the video: an animated background,
    the lecturer's own picture-in-picture. Slides change far less often."""
    import numpy as np

    signatures = [item["signature"] for item in records if "signature" in item]
    if len(signatures) < 10:
        return None
    changed = np.zeros(signatures[0].shape[1:], dtype="float32")
    previous = signatures[0].astype("float32") / 255.0
    for signature in signatures[1:]:
        current = signature.astype("float32") / 255.0
        # Grey level or edges: flickering particles mostly show up as edges.
        changed += ((np.abs(current[0] - previous[0]) > pixel_change)
                    | (np.abs(current[1] - previous[1]) > 2 * pixel_change))
        previous = current
    busy = changed / (len(signatures) - 1) > busy_fraction
    return busy if 0 < busy.mean() < 0.85 else None


def masked_boundaries(records, duration, mask, change_threshold=0.055,
                      stability_threshold=0.035, stable_samples=2, min_interval_sec=2.0):
    """Stable-change detection on the 1 Hz samples with the busy pixels ignored."""
    def samples():
        for item in records:
            signature = item["signature"].astype("float32") / 255.0
            signature[:, mask] = 0.0
            yield {"time_sec": item["time_sec"], "signature": signature}

    boundaries, _, _ = detect_stable_boundaries(
        samples(), duration, change_threshold, stability_threshold,
        stable_samples, min_interval_sec,
    )
    return boundaries


def union_boundaries(primary, extra, min_interval_sec=2.0):
    """Keep every primary boundary (it has sub-second refinement) and add the
    extra ones that are not within min_interval_sec of any kept boundary."""
    kept = sorted(primary)
    for time in sorted(extra):
        if all(abs(time - other) >= min_interval_sec for other in kept):
            kept.append(time)
            kept.sort()
    return kept


def hidden_change_splits(records, start, end, change_threshold=0.055,
                         stability_threshold=0.035, min_interval_sec=2.0,
                         static_tolerance=0.10, motion_threshold=0.15,
                         settled_threshold=0.25):
    """Split points the stable-change detector cannot confirm because one side
    of the change keeps moving (an advert clip, a camera shot of the lecturer):

    1. still -> moving: a still slide followed by footage that never holds still;
    2. moving -> settled: footage followed by a shot that only moves slightly.

    Gradual changes of a still slide (writing, reveals) are not split here."""
    import statistics

    samples = [item for item in records
               if start <= item["time_sec"] < end and "signature" in item]
    if len(samples) < 4:
        return []
    signatures = [item["signature"].astype("float32") / 255.0 for item in samples]
    times = [item["time_sec"] for item in samples]
    steps = [0.0] + [signature_distance(a, b) for a, b in zip(signatures, signatures[1:])]

    def long_enough(index, first, last):
        return (times[index] - times[first] >= min_interval_sec
                and last - times[index] >= min_interval_sec)

    def still_to_moving(first, stop, last):
        reference = signatures[first]
        to_reference = [signature_distance(reference, signatures[index])
                        for index in range(first, stop)]
        for split in range(first + 1, stop - 1):
            before = to_reference[:split - first]
            after = to_reference[split - first:]
            if (max(before) <= static_tolerance and min(after) >= change_threshold
                    and statistics.median(steps[split + 1:stop]) >= motion_threshold
                    and long_enough(split, first, last)):
                return split
        return None

    def moving_to_settled(first, stop, last):
        # The piece's last sample may already show the next state (boundary lag).
        for split in range(first + 2, stop - 2):
            if (statistics.median(steps[first + 1:split + 1]) >= motion_threshold
                    and max(steps[split + 1:stop - 1]) <= settled_threshold
                    and long_enough(split, first, last)):
                return split
        return None

    splits = []
    pieces = [(0, len(samples), end)]
    split = still_to_moving(0, len(samples), end)
    if split is not None:
        splits.append(times[split])
        pieces = [(0, split, times[split]), (split, len(samples), end)]
    for first, stop, last in pieces:
        settled = moving_to_settled(first, stop, last)
        if settled is not None:
            splits.append(times[settled])
    return sorted(splits)


def refine_from_buffer(candidate, change_threshold, stability_threshold):
    """Earliest fine-rate frame that already shows the new, stable state."""
    for timestamp, signature in candidate.get("fine", ()):
        if (signature_distance(candidate["old_signature"], signature) >= change_threshold
                and signature_distance(candidate["signature"], signature)
                <= stability_threshold):
            return timestamp
    return candidate["time_sec"]


def pick_scanned_keyframe(records, start, end):
    """Sharpest scanned sample inside [start, end), away from the edges if possible."""
    inside = [item for item in records if start <= item["time_sec"] < end]
    margin = min(0.75, (end - start) / 4)
    central = [item for item in inside
               if start + margin <= item["time_sec"] < end - margin]
    choices = central or inside
    return max(choices, key=lambda item: item["sharpness"]) if choices else None


def detect_stable_boundaries(samples, duration, change_threshold=0.055,
                             stability_threshold=0.035, stable_samples=2,
                             min_interval_sec=2.0):
    boundaries = [0.0]
    candidates = []
    reference = None
    candidate = None
    previous_time = 0.0
    sample_count = 0
    for sample in samples:
        sample_count += 1
        signature, timestamp = sample["signature"], sample["time_sec"]
        if reference is None:
            reference = signature
        changed = signature_distance(reference, signature) >= change_threshold
        eligible = timestamp - boundaries[-1] >= min_interval_sec
        if changed and eligible:
            if (candidate is None or
                    signature_distance(candidate["signature"], signature)
                    > stability_threshold):
                candidate = {
                    "time_sec": timestamp, "left_sec": previous_time,
                    "signature": signature, "old_signature": reference, "count": 1,
                    "fine": sample.get("fine", ()),
                }
            else:
                candidate["count"] += 1
            if (candidate["count"] >= stable_samples
                    and duration - candidate["time_sec"] >= min_interval_sec):
                boundaries.append(candidate["time_sec"])
                candidates.append(candidate)
                reference = candidate["signature"]
                candidate = None
        else:
            candidate = None
        previous_time = timestamp
    if sample_count == 0:
        raise ValueError("Không giải mã được frame có timestamp.")
    return boundaries + [duration], candidates, sample_count


def frame_at_time(video_path, source, time_sec):
    import av

    target = max(0.0, min(float(time_sec), source["duration_sec"] - 0.001))
    with av.open(str(video_path)) as container:
        stream = container.streams.video[0]
        origin = source["timeline_origin_sec"]
        seek_pts = int((target + origin) / float(stream.time_base))
        container.seek(seek_pts, stream=stream, backward=True, any_frame=False)
        last = None
        for frame in container.decode(stream):
            if frame.pts is None:
                continue
            timestamp = float(frame.pts * frame.time_base) - origin
            if timestamp < 0:
                continue
            last = (frame.to_image(), timestamp)
            if timestamp + 1e-6 >= target:
                return last
        if last is not None:
            return last
    raise ValueError(f"Không lấy được frame tại {time_sec:.3f}s")


def refine_boundary(video_path, source, candidate, roi, ignored_regions,
                    change_threshold, stability_threshold, samples=7):
    import numpy as np

    for target in np.linspace(candidate["left_sec"], candidate["time_sec"], samples):
        image, actual_time = frame_at_time(video_path, source, target)
        signature = frame_signature(image, roi, ignored_regions)
        if (signature_distance(candidate["old_signature"], signature) >= change_threshold
                and signature_distance(candidate["signature"], signature)
                <= stability_threshold):
            return actual_time
    return candidate["time_sec"]


def select_keyframe(video_path, source, start, end, maximum_width=1600):
    import cv2
    import numpy as np

    margin = min(0.75, (end - start) / 4)
    choices = []
    for target in (start + margin, (start + end) / 2, end - margin):
        image, actual = frame_at_time(video_path, source, target)
        if not start <= actual < end:
            continue
        gray = cv2.cvtColor(np.asarray(image), cv2.COLOR_RGB2GRAY)
        sharpness = float(cv2.Laplacian(gray, cv2.CV_64F).var())
        choices.append((sharpness, image, actual))
    if not choices:
        raise ValueError(f"Không có keyframe nằm trong [{start}, {end}).")
    sharpness, image, actual = max(choices, key=lambda item: item[0])
    image.thumbnail((maximum_width, maximum_width))
    return image, actual, sharpness


def create_contact_sheet(directory, intervals, limit=24):
    from PIL import Image, ImageDraw

    selected = intervals[:limit]
    columns, cell_width, cell_height = 4, 320, 210
    sheet = Image.new("RGB", (columns * cell_width,
                             max(1, math.ceil(len(selected) / columns)) * cell_height),
                      "#f5f1e9")
    draw = ImageDraw.Draw(sheet)
    for index, item in enumerate(selected):
        with Image.open(relative_asset(directory, item["image_path"])) as image:
            image = image.convert("RGB")
            image.thumbnail((cell_width - 12, 174))
            x, y = (index % columns) * cell_width + 6, (index // columns) * cell_height
            sheet.paste(image, (x, y + 6))
            label = (f"{item['sequence']:03d} | {timestamp_label(item['start_sec'])}"
                     f" - {timestamp_label(item['end_sec'])}")
            draw.text((x, y + 185), label, fill="#241f18")
    sheet.save(Path(directory) / "contact_sheet.jpg", quality=90)


def reveal_view(image, roi=(0, 0, 1, 1), ignored_regions=()):
    """Grayscale ROI at a fixed size, for comparing content between visual states."""
    import cv2
    import numpy as np

    pixels = np.asarray(image.convert("RGB")).copy()
    height, width = pixels.shape[:2]
    for left, top, right, bottom in ignored_regions:
        pixels[int(top * height):int(bottom * height),
               int(left * width):int(right * width)] = 127
    left, top, right, bottom = roi
    pixels = pixels[int(top * height):max(int(bottom * height), 1),
                    int(left * width):max(int(right * width), 1)]
    gray = cv2.cvtColor(pixels, cv2.COLOR_RGB2GRAY)
    return cv2.resize(gray, (384, 216), interpolation=cv2.INTER_AREA)


def content_mask(view):
    import cv2

    # Text and drawings differ from their local background; a wide median keeps
    # the slide background and removes thin strokes.
    return cv2.absdiff(view, cv2.medianBlur(view, 41)) > 28


def persistent_template(views, fraction=0.55):
    """Content present in most states (logo, title bar, frame) is not evidence."""
    import numpy as np

    if len(views) < 6:
        return np.zeros(views[0].shape, dtype=bool) if views else None
    indices = np.unique(np.linspace(0, len(views) - 1, min(24, len(views))).astype(int))
    return np.mean([content_mask(views[index]) for index in indices], axis=0) >= fraction


def reveal_evidence(previous, current, template):
    """How much old content survives at the same place, and how much was added."""
    import cv2
    import numpy as np

    old = content_mask(previous) & ~template
    new = content_mask(current) & ~template
    kernel = np.ones((3, 3), np.uint8)
    # Allow one-pixel shifts and compression noise around each old stroke.
    low = cv2.erode(current, kernel).astype(np.int16) - 22
    high = cv2.dilate(current, kernel).astype(np.int16) + 22
    values = previous.astype(np.int16)
    preserved = (values >= low) & (values <= high)
    old_pixels = int(np.count_nonzero(old))
    return {
        "old_content_pixels": old_pixels,
        "content_retained": float(np.count_nonzero(old & preserved) / max(1, old_pixels)),
        "content_growth": float(np.count_nonzero(new) / max(1, old_pixels)),
        "image_change": float(np.mean(cv2.absdiff(previous, current) > 32)),
    }


def ocr_words(text):
    return set(re.findall(r"[^\W_]{2,}", str(text).casefold(), flags=re.UNICODE))


def common_ocr_words(texts):
    """Words on most states of a video (channel name, author, footer) are not content."""
    from collections import Counter

    counts = Counter(word for text in texts for word in ocr_words(text))
    return {word for word, count in counts.items() if count >= max(12, 0.55 * len(texts))}


def text_evidence(previous_text, current_text, common):
    old, new = ocr_words(previous_text) - common, ocr_words(current_text) - common
    shared = len(old & new)
    return {"old_words": len(old), "new_words": len(new), "shared_words": shared,
            "text_containment": shared / max(1, min(len(old), len(new)))}


def reveal_decision(previous, current, anchor, state, *, minimum_pixels=250,
                    retained_threshold=0.85, change_limit=0.25, growth_threshold=0.9,
                    anchor_threshold=0.6, non_slide_threshold=0.2):
    """Merge only when the new state keeps the old content and adds to it.

    previous/current/anchor index the per-video state arrays in ``state``: views,
    the persistent-content mask, and optionally OCR texts and slide probabilities.
    """
    views, mask = state["views"], state["mask"]
    texts, p_slides, common = state.get("texts"), state.get("p_slides"), state.get("common", ())
    evidence = reveal_evidence(views[previous], views[current], mask)
    if p_slides is not None:
        evidence["p_slide_previous"] = p_slides[previous]
        evidence["p_slide_current"] = p_slides[current]
        # A camera shot of the lecturer is never part of a slide build.
        if min(p_slides[previous], p_slides[current]) < non_slide_threshold:
            return "split_non_slide", evidence
    text = None
    if texts is not None:
        text = text_evidence(texts[previous], texts[current], common)
        evidence.update(text)
        visual_build = (evidence["content_retained"] >= 0.88
                        and evidence["image_change"] <= change_limit)
        # Same layout, different words (e.g. remember -> regret): a new slide.
        if (text["old_words"] >= 4 and text["new_words"] >= 4
                and text["text_containment"] < 0.55 and not visual_build):
            return "split_text_replaced", evidence
    if (text is not None and text["shared_words"] >= 4 and text["text_containment"] >= 0.78
            and evidence["image_change"] <= 0.30):
        # OCR keeps the old words even when a moving presenter lowers pixel retention.
        decision = "merge_text_retained"
    elif evidence["old_content_pixels"] < minimum_pixels:
        return "split_little_content", evidence
    elif (evidence["content_retained"] >= retained_threshold
            and evidence["image_change"] <= change_limit
            and evidence["content_growth"] >= growth_threshold):
        decision = "merge_content_kept"
    else:
        return "split_content_changed", evidence
    if anchor != previous:
        # Prevent chained merges: A~B and B~C while C no longer shows A's core.
        first = reveal_evidence(views[anchor], views[current], mask)
        evidence["anchor_retained"] = first["content_retained"]
        text_lost = False
        if texts is not None:
            core = text_evidence(texts[anchor], texts[current], common)
            text_lost = (core["old_words"] >= 4 and core["new_words"] >= 4
                         and core["shared_words"] / core["old_words"] < 0.40)
        if first["content_retained"] < anchor_threshold or (
                text_lost and first["content_retained"] < 0.75):
            return "split_anchor_lost", evidence
    return decision, evidence


def group_reveal_states(views, texts=None, p_slides=None, **thresholds):
    """Return [[state indices]] where each group is one slide revealed over time."""
    if not views:
        return [], []
    state = {"views": views, "mask": persistent_template(views),
             "texts": texts, "p_slides": p_slides,
             "common": common_ocr_words(texts) if texts is not None else set()}
    groups, audit = [[0]], []
    for index in range(1, len(views)):
        group = groups[-1]
        decision, evidence = reveal_decision(group[-1], index, group[0], state, **thresholds)
        audit.append({"previous_state": group[-1], "current_state": index,
                      "decision": decision,
                      **{key: round(value, 4) for key, value in evidence.items()}})
        if decision.startswith("merge_"):
            group.append(index)
        else:
            groups.append([index])
    return groups, audit


SLIDE_CLASSIFIER_URL = (
    "https://huggingface.co/HHousen/lecture2notes/resolve/main/"
    "Slide%20Classifier%20Median%20Weights/three-category/epoch%3D8.ckpt"
)


def load_slide_classifier(checkpoint=None, cache_dir="/kaggle/working/_models"):
    """Lecture2Notes ResNet-34 (other / presenter_slide / slide); returns p_slide(images)."""
    import urllib.request
    import torch
    import torch.nn as nn
    import torchvision.models as models
    import torchvision.transforms as transforms

    if checkpoint is None:
        found = sorted(Path("/kaggle/input").rglob("epoch=8.ckpt"))
        checkpoint = found[0] if found else Path(cache_dir) / "lecture2notes_epoch8.ckpt"
    checkpoint = Path(checkpoint)
    if not checkpoint.is_file():
        checkpoint.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(SLIDE_CLASSIFIER_URL, checkpoint)
    data = torch.load(checkpoint, map_location="cpu", weights_only=False)
    hp = data.get("hyper_parameters", data.get("hparams", {}))
    hp = hp if isinstance(hp, dict) else vars(hp)
    classes = list(hp.get("classes", []))
    if set(classes) != {"other", "presenter_slide", "slide"} or hp.get("arch") != "resnet34":
        raise RuntimeError(f"Checkpoint không phải ResNet-34 ba lớp: {classes}, {hp.get('arch')}")

    class ConcatPool(nn.Module):
        def forward(self, x):
            return torch.cat([nn.functional.adaptive_max_pool2d(x, 1),
                              nn.functional.adaptive_avg_pool2d(x, 1)], dim=1)

    base = models.resnet34(weights=None)
    if hp.get("feature_extract") == "advanced":
        model = nn.Sequential(
            nn.Sequential(*list(base.children())[:-2]), ConcatPool(), nn.Flatten(),
            nn.BatchNorm1d(1024), nn.Dropout(0.25), nn.Linear(1024, 512),
            nn.ReLU(inplace=True), nn.BatchNorm1d(512), nn.Dropout(0.5),
            nn.Linear(512, len(classes)),
        )
    else:
        base.fc = nn.Linear(base.fc.in_features, len(classes))
        model = base
    if hp.get("relu_to_mish"):
        def replace(module):
            for name, child in module.named_children():
                if isinstance(child, nn.ReLU):
                    setattr(module, name, nn.Mish())
                else:
                    replace(child)
        replace(model)
    prefix = "classification_model."
    model.load_state_dict({key[len(prefix):]: value for key, value in data["state_dict"].items()
                           if key.startswith(prefix)}, strict=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device).eval()
    size = int(hp.get("input_size", 224))
    # The authors' inference transform: resize the short side, then center crop.
    preprocess = transforms.Compose([
        transforms.Resize(size), transforms.CenterCrop(size), transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    positive = [classes.index("slide"), classes.index("presenter_slide")]

    def p_slide(images, batch_size=32):
        values = []
        for start in range(0, len(images), batch_size):
            batch = torch.stack([preprocess(image.convert("RGB"))
                                 for image in images[start:start + batch_size]]).to(device)
            with torch.inference_mode():
                probabilities = torch.softmax(model(batch), dim=1)[:, positive].sum(dim=1)
            values += [float(value) for value in probabilities.cpu()]
        return values

    return p_slide, file_sha256(checkpoint)


def tesseract_languages():
    if not shutil.which("tesseract"):
        raise RuntimeError("Cần tesseract (apt-get install tesseract-ocr tesseract-ocr-vie).")
    listed = run_command(["tesseract", "--list-langs"]).lower().split()
    return "+".join(language for language in ("vie", "eng") if language in listed) or "eng"


def tesseract_text(image, languages, roi=(0, 0, 1, 1)):
    """Tesseract sparse-text OCR; keeps words with confidence >= 35."""
    import csv
    import io
    import cv2
    import numpy as np

    pixels = np.asarray(image.convert("L"))
    height, width = pixels.shape
    left, top, right, bottom = roi
    pixels = pixels[int(top * height):int(bottom * height), int(left * width):int(right * width)]
    if pixels.shape[1] < 720:
        pixels = cv2.resize(pixels, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
    result = subprocess.run(
        ["tesseract", "stdin", "stdout", "-l", languages, "--psm", "11", "tsv"],
        input=cv2.imencode(".png", pixels)[1].tobytes(), capture_output=True, timeout=60,
    )
    if result.returncode:
        raise RuntimeError(result.stderr.decode(errors="replace")[-500:])
    rows = csv.DictReader(io.StringIO(result.stdout.decode("utf-8", "replace")),
                          delimiter="\t", quoting=csv.QUOTE_NONE)
    words = []
    for row in rows:
        text = (row.get("text") or "").strip()
        try:
            confidence = float(row.get("conf") or -1)
        except ValueError:
            confidence = -1
        if confidence >= 35 and len(text) >= 2:
            words.append(text)
    return " ".join(words)


def create_reveal_sheet(directory, intervals, limit=12):
    """One row per merged slide: every reveal state from first to final."""
    from PIL import Image, ImageDraw

    merged = [item for item in intervals if len(item["reveal_states"]) > 1][:limit]
    if not merged:
        return None
    columns = min(6, max(len(item["reveal_states"]) for item in merged))
    cell_width, cell_height = 260, 175
    sheet = Image.new("RGB", (columns * cell_width, len(merged) * cell_height), "#f5f1e9")
    draw = ImageDraw.Draw(sheet)
    for row, item in enumerate(merged):
        states = item["reveal_states"]
        # Keep the first and the final states when a group has too many to show.
        shown = states if len(states) <= columns else states[:columns - 1] + states[-1:]
        for column, state in enumerate(shown):
            with Image.open(relative_asset(directory, state["image_path"])) as image:
                image = image.convert("RGB")
                image.thumbnail((cell_width - 10, 140))
                x, y = column * cell_width + 5, row * cell_height
                sheet.paste(image, (x, y + 5))
            draw.text((x, y + 150), f"{item['sequence']:03d}.{state['state']} "
                      f"{timestamp_label(state['start_sec'])}", fill="#241f18")
    sheet.save(Path(directory) / "reveal_sheet.jpg", quality=90)
    return "reveal_sheet.jpg"


def run_slide_detection(video_path, output_root, video_id="L25_V001", *,
                        sample_hz=1.0, roi=(0, 0, 1, 1), ignored_regions=(),
                        change_threshold=0.055, stability_threshold=0.035,
                        stable_samples=2, min_interval_sec=2.0,
                        manual_boundaries=None, merge_reveals=True,
                        reveal_thresholds=None, slide_classifier=None, merge_ocr=False,
                        scan="single_pass", fine_hz=8.0, skip_nonref=True,
                        split_hidden_changes=True, mask_motion=True,
                        ad_scorer=None, ad_options=None):
    """slide_classifier: None, or (p_slide function, checkpoint sha256) from
    load_slide_classifier(). merge_ocr: compare Tesseract words between states.
    ad_scorer: None, or load_ad_scorer() of adfilter.py: ad segments are cut out of
    the slide intervals as their own segments (content_kind "ad"; single_pass only).
    ad_options: extra apply_ad_filter() arguments, e.g. {"mode": "flag"} to only flag them.
    scan: "single_pass" decodes once with no seeks; "seek" is the original method."""
    from PIL import Image

    if sample_hz <= 0 or stable_samples < 2 or min_interval_sec <= 0:
        raise ValueError("sample_hz/min_interval_sec phải > 0; stable_samples >= 2.")
    for box in (roi, *ignored_regions):
        if not (len(box) == 4 and 0 <= box[0] < box[2] <= 1
                and 0 <= box[1] < box[3] <= 1):
            raise ValueError(f"Vùng chuẩn hóa [left, top, right, bottom] sai: {box}")
    if scan not in {"single_pass", "seek"}:
        raise ValueError("SCAN_MODE phải là 'single_pass' hoặc 'seek'.")
    source = probe_video(video_path)
    directory = begin_stage(output_root, 1)
    records = []
    hidden_split_count = masked_split_count = 0
    motion_mask_fraction = None
    if manual_boundaries is None and scan == "single_pass":
        def run_scan(skip):
            records.clear()
            return detect_stable_boundaries(
                scan_frames(video_path, source, directory, sample_hz, fine_hz,
                            roi, ignored_regions, skip_nonref=skip, records=records),
                source["duration_sec"], change_threshold, stability_threshold,
                stable_samples, min_interval_sec,
            )
        try:
            boundaries, candidates, sample_count = run_scan(skip_nonref)
        except ValueError:
            if not skip_nonref:
                raise
            # Some streams have no reference frames left after skipping.
            boundaries, candidates, sample_count = run_scan(False)
        refined = [0.0]
        for candidate in candidates:
            boundary = refine_from_buffer(candidate, change_threshold, stability_threshold)
            refined.append(max(refined[-1] + min_interval_sec,
                               min(boundary, candidate["time_sec"])))
        boundaries = refined + [source["duration_sec"]]
        if mask_motion:
            mask = motion_mask(records)
            if mask is not None:
                motion_mask_fraction = round(float(mask.mean()), 4)
                before = len(boundaries)
                boundaries = union_boundaries(boundaries, masked_boundaries(
                    records, source["duration_sec"], mask, change_threshold,
                    stability_threshold, stable_samples, min_interval_sec,
                ), min_interval_sec)
                masked_split_count = len(boundaries) - before
        if split_hidden_changes:
            checked = [boundaries[0]]
            for start, end in zip(boundaries, boundaries[1:]):
                checked += hidden_change_splits(
                    records, start, end, change_threshold, stability_threshold,
                    min_interval_sec,
                ) + [end]
            hidden_split_count = len(checked) - len(boundaries)
            boundaries = checked
    elif manual_boundaries is None:
        boundaries, candidates, sample_count = detect_stable_boundaries(
            sampled_frames(video_path, source, sample_hz, roi, ignored_regions),
            source["duration_sec"], change_threshold, stability_threshold,
            stable_samples, min_interval_sec,
        )
        refined = [0.0]
        for candidate in candidates:
            boundary = refine_boundary(
                video_path, source, candidate, roi, ignored_regions,
                change_threshold, stability_threshold,
            )
            # Do not create tiny intervals while refining a transition.
            refined.append(max(refined[-1] + min_interval_sec,
                               min(boundary, candidate["time_sec"])))
        boundaries = refined + [source["duration_sec"]]
    else:
        boundaries = [float(time) for time in manual_boundaries]
        if boundaries != sorted(set(boundaries)):
            raise ValueError("MANUAL_BOUNDARIES_SEC phải tăng dần, không trùng.")
        if (not boundaries or abs(boundaries[0]) > 1e-6
                or abs(boundaries[-1] - source["duration_sec"]) > 0.03):
            raise ValueError("Ranh giới thủ công phải bắt đầu 0 và kết thúc ở duration.")
        sample_count = 0

    # Each visual state gets its own sharp keyframe before states are grouped.
    states, views = [], []
    for index, (start, end) in enumerate(zip(boundaries, boundaries[1:])):
        scanned = pick_scanned_keyframe(records, start, end)
        if scanned is not None:
            with Image.open(directory / scanned["image_path"]) as opened:
                image = opened.convert("RGB")
            keyframe_time, sharpness = scanned["time_sec"], scanned["sharpness"]
        else:
            image, keyframe_time, sharpness = select_keyframe(video_path, source, start, end)
        name = f"keyframes/states/state_{index:05d}.jpg"
        path = directory / name
        path.parent.mkdir(parents=True, exist_ok=True)
        image.save(path, quality=95)
        states.append({"start_sec": round(start, 6), "end_sec": round(end, 6),
                       "keyframe_time_sec": round(keyframe_time, 6),
                       "image_path": name, "sharpness_score": round(sharpness, 3)})
        views.append(reveal_view(image, roi, ignored_regions))
        if index % 20 == 0:
            print(f"Keyframe {index + 1}/{len(boundaries) - 1}: {start:.2f}s")
    p_slides = texts = None
    if slide_classifier is not None or merge_ocr:
        languages = tesseract_languages() if merge_ocr else None
        if merge_ocr:
            texts = []
        images = []
        for index, item in enumerate(states):
            with Image.open(directory / item["image_path"]) as opened:
                image = opened.convert("RGB")
            if merge_ocr:
                texts.append(tesseract_text(image, languages, roi))
                item["ocr_text"] = texts[-1]
            images.append(image)
            if len(images) == 64 or index == len(states) - 1:
                if slide_classifier is not None:
                    p_slides = (p_slides or []) + slide_classifier[0](images)
                images = []
        if p_slides is not None:
            for item, value in zip(states, p_slides):
                item["p_slide"] = round(value, 4)
    if merge_reveals and manual_boundaries is None:
        groups, reveal_audit = group_reveal_states(
            views, texts, p_slides, **(reveal_thresholds or {}))
    else:
        groups, reveal_audit = [[index] for index in range(len(states))], []
    decisions = {item["current_state"]: item["decision"] for item in reveal_audit}

    method = ("manual" if manual_boundaries is not None
              else "single_pass_stable_visual_change" if scan == "single_pass"
              else "stable_visual_change_with_pts_refinement")
    intervals = []
    for index, group in enumerate(groups):
        members = [states[position] for position in group]
        # The final state of a progressive reveal shows the complete slide.
        final = members[-1]
        name = f"keyframes/seg_{index:05d}.jpg"
        shutil.copyfile(directory / final["image_path"], directory / name)
        intervals.append({
            "segment_id": f"{video_id}:seg_{index:05d}", "sequence": index,
            "start_sec": members[0]["start_sec"], "end_sec": final["end_sec"],
            "keyframe_time_sec": final["keyframe_time_sec"],
            "image_path": name, "image_sha256": file_sha256(directory / name),
            "content_kind": "unclassified",
            "sharpness_score": final["sharpness_score"],
            **({"p_slide": final["p_slide"]} if "p_slide" in final else {}),
            "boundary_method": method + ("+progressive_reveal_merge" if len(group) > 1 else ""),
            "reveal_states": [
                {"state": number, "source_state": position, **states[position],
                 "decision": decisions.get(position, "first_state")}
                for number, position in enumerate(group)
            ],
        })
    ad_report = None
    if ad_scorer is not None and records:
        intervals, ad_report = apply_ad_filter(
            intervals, records, directory, video_id, source["duration_sec"], boundaries,
            ad_scorer, min_interval_sec=min_interval_sec, **(ad_options or {}))
        write_json(directory / "ad_filter.json", ad_report)
    shutil.rmtree(directory / "_scan", ignore_errors=True)
    verify_intervals(intervals, source["duration_sec"])
    write_json(directory / "intervals.json", intervals)
    write_json(directory / "reveal_audit.json", reveal_audit)
    create_contact_sheet(directory, intervals)
    reveal_sheet = create_reveal_sheet(directory, intervals)
    parameters = {
        "sample_hz": sample_hz, "roi": roi, "ignored_regions": ignored_regions,
        "change_threshold": change_threshold, "stability_threshold": stability_threshold,
        "stable_samples": stable_samples, "min_interval_sec": min_interval_sec,
        "manual_boundaries_sec": manual_boundaries,
        "merge_reveals": bool(merge_reveals and manual_boundaries is None),
        "reveal_thresholds": reveal_thresholds or {},
        "slide_classifier_sha256": slide_classifier[1] if slide_classifier else None,
        "merge_ocr": bool(merge_ocr),
        "scan": scan, "fine_hz": fine_hz, "skip_nonref": skip_nonref,
        "split_hidden_changes": bool(split_hidden_changes and scan == "single_pass"),
        "mask_motion": bool(mask_motion and scan == "single_pass"),
        "packages": package_versions(["av", "numpy", "opencv-python-headless", "Pillow"]),
        "ad_filter": {key: value for key, value in ad_report.items()
                      if key not in {"sample_scores", "sample_times"}}
                     if ad_report else None,
    }
    manifest = finish_stage(directory, 1, video_id, source, parameters, extra={
        "sample_count": sample_count, "interval_count": len(intervals),
        "visual_state_count": len(states), "hidden_split_count": hidden_split_count,
        "masked_split_count": masked_split_count, "motion_mask_fraction": motion_mask_fraction,
        "ad_segment_count": sum(item["content_kind"] == "ad" for item in intervals),
        "ad_flagged_count": sum(bool(item.get("ad_flagged")) for item in intervals),
        "merged_reveal_count":
            sum(len(group) > 1 for group in groups),
        "reveal_sheet": reveal_sheet,
        "semantic_classification": "pending_stage_02",
    })
    return directory, manifest, intervals


In [ ]:
"""Ad filter: score each 1 Hz sample "lesson or ad", turn the scores into ad intervals and
cut them out of the slide intervals as their own segments (content_kind "ad").

Stage 01 calls apply_ad_filter() with the 1 Hz samples it already decoded. Later stages
skip "ad" segments: no OCR (02), their words stay with the ad (04, midpoint rule as for
slides), nothing is indexed (05).

mode "flag" (content filter phase 1): the ad detector no longer removes content. Ad time
ranges still become their own segments, but as ordinary segments with ad_flagged=True: the
OCR gate decides OCR (02), the speech is kept (04), and only the index leaves them out (05).
A guard drops every flag when the detector looks unreliable for the channel: more ad runs or
a larger share of the video than any L25 video has (L25 gold: 3-4 runs, 7.6-24.5 %).
"""

AD_LESSON_PROMPTS = [
    "a slide of a lecture presentation with text",
    "a lecture slide with a diagram, chart or formula",
    "a teacher presenting a lesson next to a slide",
    "the title slide of a lesson",
    "a page of a document or an exam with questions",
]
AD_PROMPTS = [
    "a video advertisement",
    "a promotional video of a school or university",
    "a logo animation",
    "students walking on a school campus",
    "people in a commercial",
    "a building exterior",
]


def ad_runs(mask):
    """[start, end) index runs of a 0/1 sequence where it is 1."""
    runs, index = [], 0
    while index < len(mask):
        if mask[index]:
            end = index
            while end < len(mask) and mask[end]:
                end += 1
            runs.append([index, end])
            index = end
        else:
            index += 1
    return runs


def ad_mask(scores, threshold=0.5, smooth=5, min_ad=5, max_gap=2):
    """Per-sample ad mask: moving average, threshold, fill short gaps between two ads,
    drop ads shorter than min_ad samples except a short closing logo at the very end."""
    import numpy as np

    scores = np.asarray(scores, dtype="float64")
    if smooth > 1 and len(scores):
        # edge padding: a short closing logo is not diluted by zeros past the end of the video
        padded = np.pad(scores, (smooth // 2, smooth - 1 - smooth // 2), mode="edge")
        scores = np.convolve(padded, np.ones(smooth) / smooth, mode="valid")
    mask = (scores >= threshold).astype(int)
    for start, end in ad_runs(1 - mask):
        if 0 < start and end < len(mask) and end - start <= max_gap:
            mask[start:end] = 1
    for start, end in ad_runs(mask):
        if end - start < min_ad and not (end == len(mask) and end - start >= 2):
            mask[start:end] = 0
    return mask


def model_features(output):
    """Embedding tensor from get_image_features/get_text_features (transformers 4 or 5)."""
    import torch

    return output if torch.is_tensor(output) else output.pooler_output


def load_siglip(model_id, device="cuda"):
    """SigLIP loaded once per session (shared by the ad filter and the OCR gate).
    Returns (embed_images(images), embed_texts(texts), logit_scale, logit_bias)."""
    def load():
        import torch
        from transformers import AutoModel, AutoProcessor

        dtype = torch.float16 if str(device).startswith("cuda") else torch.float32
        model = AutoModel.from_pretrained(model_id, torch_dtype=dtype).to(device).eval()
        processor = AutoProcessor.from_pretrained(model_id)

        def embed_images(images):
            with torch.no_grad():
                pixels = processor(images=images, return_tensors="pt")["pixel_values"].to(device, dtype)
                out = model_features(model.get_image_features(pixel_values=pixels)).float()
            return torch.nn.functional.normalize(out, dim=-1).cpu().numpy()

        def embed_texts(texts):
            with torch.no_grad():
                tokens = processor(text=texts, padding="max_length", return_tensors="pt").to(device)
                out = model_features(model.get_text_features(**tokens)).float()
            return torch.nn.functional.normalize(out, dim=-1).cpu().numpy()

        return embed_images, embed_texts, model.logit_scale.exp().item(), model.logit_bias.item()

    return shared_model(("siglip", model_id, str(device)), load)


def zero_shot(embed_images, embed_texts, scale, bias, positive, negative):
    """P(positive prompts) for each image: softmax over all prompts, summed over the positive ones."""
    import numpy as np

    text = embed_texts(list(positive) + list(negative))

    def score(images):
        logits = embed_images(images) @ text.T * scale + bias
        logits = logits - logits.max(1, keepdims=True)
        probs = np.exp(logits) / np.exp(logits).sum(1, keepdims=True)
        return probs[:, :len(positive)].sum(1).tolist()

    return score


def load_ad_scorer(mode="student", device="cuda", student_path=None,
                   teacher_model="google/siglip-so400m-patch14-384"):
    """Returns (score(images) -> list of P(ad), description). mode "teacher": SigLIP
    zero-shot with the prompts above; "student": logistic regression (student_path, an
    .npz from the ad filter experiments) on the image embeddings of a small SigLIP."""
    import numpy as np

    if mode not in {"student", "teacher"}:
        raise ValueError("AD_FILTER phải là 'student', 'teacher' hoặc None.")
    student = dict(np.load(student_path, allow_pickle=False)) if mode == "student" else None
    if student and bool(student.get("with_motion", False)):
        raise ValueError("Pipeline dùng model trò chỉ có ảnh (ad_student.npz), không phải bản có chuyển động.")
    model_id = str(student["backbone"]) if student else teacher_model
    embed_images, embed_texts, scale, bias = load_siglip(model_id, device)
    if mode == "teacher":
        score = zero_shot(embed_images, embed_texts, scale, bias, AD_PROMPTS, AD_LESSON_PROMPTS)
    else:
        def score(images):
            logits = embed_images(images) @ student["coef"] + float(student["intercept"][0])
            return (1 / (1 + np.exp(-logits))).tolist()
    threshold = float(student["threshold"]) if student else 0.5
    return score, {"mode": mode, "model": model_id, "threshold": threshold,
                   "student_sha256": file_sha256(student_path) if student else None}


# OCR gate: a keyframe goes to OCR only if it shows readable lesson content. A lecturer
# talking to the camera, an intro logo or a blank screen is kept as a segment (its speech
# stays in the lesson) but is not read by the slow OCR models.
CONTENT_PROMPTS = [
    "a slide of a lecture presentation with text",
    "a lecture slide with a diagram, chart, table or formula",
    "a page of a document or an exam with questions",
    "a whiteboard or blackboard with handwriting",
]
NO_CONTENT_PROMPTS = [
    "a person talking to the camera",
    "a teacher speaking in a studio",
    "a logo or a title animation",
    "a blank screen",
]


def load_ocr_gate(device="cuda", model_id="google/siglip-base-patch16-224", threshold=0.10, views="frame"):
    """Returns (check(images) -> list of P(readable content), description). views: "frame"
    scores the whole keyframe; "frame+crops" also looks at its slide region, quarters and centre
    before skipping (visual.gate_keyframe), so it never skips more than "frame"."""
    if views not in {"frame", "frame+crops"}:
        raise ValueError("OCR_GATE_VIEWS phải là 'frame' hoặc 'frame+crops'.")
    embed_images, embed_texts, scale, bias = load_siglip(model_id, device)
    check = zero_shot(embed_images, embed_texts, scale, bias, CONTENT_PROMPTS, NO_CONTENT_PROMPTS)
    return check, {"model": model_id, "threshold": threshold, "views": views}


def snap(time_sec, boundaries, tolerance=1.5):
    """Move an ad edge (from 1 Hz samples) to the nearest exact state boundary of stage 01."""
    nearest = min(boundaries, key=lambda item: abs(item - time_sec))
    return nearest if abs(nearest - time_sec) <= tolerance else time_sec


def ad_time_intervals(sample_times, mask, duration, boundaries):
    intervals = []
    for start, end in ad_runs(mask):
        a = snap(sample_times[start], boundaries)
        b = duration if end >= len(sample_times) else snap(sample_times[end], boundaries)
        if intervals and a <= intervals[-1][1]:
            intervals[-1][1] = max(intervals[-1][1], b)
        elif b > a:
            intervals.append([a, b])
    return intervals


def split_ad_intervals(intervals, ads, min_interval_sec=2.0):
    """Cut ad time ranges out of slide intervals. Returns [start, end, source] pieces in time
    order; source is the index of the slide interval, or None for an ad. A lesson remnant
    shorter than min_interval_sec next to an ad is absorbed by that ad."""
    cuts = sorted({item["start_sec"] for item in intervals} | {intervals[-1]["end_sec"]}
                  | {t for a, b in ads for t in (a, b) if intervals[0]["start_sec"] < t < intervals[-1]["end_sec"]})
    pieces = []
    for start, end in zip(cuts, cuts[1:]):
        middle = (start + end) / 2
        if any(a <= middle < b for a, b in ads):
            source = None
        else:
            source = next(i for i, item in enumerate(intervals) if item["start_sec"] <= middle < item["end_sec"])
        if pieces and pieces[-1][2] == source:
            pieces[-1][1] = end
        else:
            pieces.append([start, end, source])
    changed = True
    while changed:
        changed = False
        for i, (start, end, source) in enumerate(pieces):
            next_to_ad = (i > 0 and pieces[i - 1][2] is None) or (i + 1 < len(pieces) and pieces[i + 1][2] is None)
            if source is not None and end - start < min_interval_sec and next_to_ad:
                pieces[i][2] = None
                changed = True
        merged = []
        for piece in pieces:
            if merged and merged[-1][2] is None and piece[2] is None:
                merged[-1][1] = piece[1]
            else:
                merged.append(piece)
        pieces = merged
    return pieces


def kmeans(vectors, k, iterations=25, seed=0):
    """Plain k-means on unit vectors (cosine), k-means++ start. Returns (labels, centres)."""
    import numpy as np

    rng = np.random.default_rng(seed)
    k = min(k, len(vectors))
    centres = [vectors[rng.integers(len(vectors))]]
    for _ in range(1, k):
        distance = np.clip(1 - np.max(vectors @ np.array(centres).T, axis=1), 0, None)
        centres.append(vectors[rng.choice(len(vectors), p=distance / distance.sum())] if distance.sum() > 0
                       else vectors[rng.integers(len(vectors))])
    centres = np.array(centres)
    for _ in range(iterations):
        labels = np.argmax(vectors @ centres.T, axis=1)
        for index in range(k):
            members = vectors[labels == index]
            if len(members):
                mean = members.mean(0)
                centres[index] = mean / (np.linalg.norm(mean) + 1e-9)
    return np.argmax(vectors @ centres.T, axis=1), centres


def outlier_scores(vectors, changes, *, k=6, min_share=0.05, min_runs=3, run_sec=5,
                   change_cut=0.15, weight_change=1.0, scale=1.5, offset=3.0):
    """Content filter A: P(ad) per 1 Hz sample from the video alone, no learned ad model.

    vectors: unit image embeddings (SigLIP), changes: picture change to the previous sample.
    Lecture looks = k-means clusters that hold >= min_share of the time AND come back in >=
    min_runs separate runs of >= run_sec (a lecture look recurs all over the video; an ad is a
    block). Distance to the nearest lecture centre is standardised inside the video (median /
    MAD), the share of picture cuts in a 5 s window adds weight_change, and a logistic turns
    the sum into a score for ad_mask(). Returns (scores, details)."""
    import numpy as np

    vectors = np.asarray(vectors, dtype="float32")
    vectors /= np.linalg.norm(vectors, axis=1, keepdims=True) + 1e-9
    labels, centres = kmeans(vectors, k)
    lecture = []
    for index in range(len(centres)):
        mask = labels == index
        runs = [len(run) for run in np.split(mask, np.flatnonzero(np.diff(mask.astype(int))) + 1) if run[0]]
        if mask.mean() >= min_share and sum(length >= run_sec for length in runs) >= min_runs:
            lecture.append(index)
    if not lecture:   # no recurring look: fall back to the largest cluster
        lecture = [int(np.bincount(labels).argmax())]
    distance = 1 - np.max(vectors @ centres[lecture].T, axis=1)
    median = float(np.median(distance))
    mad = float(np.median(np.abs(distance - median))) * 1.4826 or 1e-6
    z = (distance - median) / mad
    cuts = (np.asarray(changes, dtype="float32") > change_cut).astype("float32")
    cut_rate = np.convolve(cuts, np.ones(5) / 5, mode="same")
    logit = (z + weight_change * 10 * cut_rate - offset) * scale
    scores = 1 / (1 + np.exp(-logit))
    return scores.tolist(), {"lecture_clusters": len(lecture), "clusters": len(centres)}


def load_outlier_scorer(device="cuda", model_id="google/siglip-base-patch16-224", threshold=0.5, **options):
    """Ad scorer for apply_ad_filter() that compares each sample with the rest of the video
    (outlier_scores()): score(images) returns embeddings, score.video_scores(embeddings,
    records) the per-sample P(ad) once the whole video is seen."""
    import numpy as np

    embed_images, _, _, _ = load_siglip(model_id, device)

    def score(images):
        return list(embed_images(images))

    def video_scores(vectors, records):
        # stage-01 records keep the signature as uint8 (x 255)
        changes = [0.0] + [float(signature_distance(a["signature"].astype("float32") / 255.0,
                                                    b["signature"].astype("float32") / 255.0))
                           if "signature" in a and "signature" in b else 0.0
                           for a, b in zip(records, records[1:])]
        values, _ = outlier_scores(np.array(vectors), changes, **options)
        return values

    score.video_scores = video_scores
    return score, {"mode": "outlier", "model": model_id, "threshold": threshold, "options": options,
                   "student_sha256": None}


def ad_guard(ads, duration, max_runs=6, max_share=0.30):
    """Reason to distrust the ad intervals of one video, or None."""
    share = sum(b - a for a, b in ads) / max(duration, 1e-9)
    if len(ads) > max_runs:
        return f"{len(ads)} ad runs > {max_runs}"
    if share > max_share:
        return f"ads cover {share:.0%} of the video > {max_share:.0%}"
    return None


def apply_ad_filter(intervals, records, directory, video_id, duration, state_boundaries,
                    scorer, *, smooth=5, min_ad=5, max_gap=2, min_interval_sec=2.0,
                    batch_size=64, mode="cut", max_runs=6, max_share=0.30):
    """intervals: stage-01 intervals (before writing). records: the 1 Hz samples of the scan
    (time_sec, image_path in directory). mode: "cut" (ad segments, content_kind "ad") or
    "flag" (ordinary segments with ad_flagged, see the module docstring).
    Returns (new intervals, ad report)."""
    if mode not in {"cut", "flag"}:
        raise ValueError("AD_MODE phải là 'cut' hoặc 'flag'.")
    from PIL import Image

    score, info = scorer
    times, scores = [item["time_sec"] for item in records], []
    for start in range(0, len(records), batch_size):
        images = []
        for item in records[start:start + batch_size]:
            with Image.open(Path(directory) / item["image_path"]) as opened:
                images.append(opened.convert("RGB"))
        scores += score(images)
    if hasattr(score, "video_scores"):   # outlier scorer: needs the whole video at once
        scores = score.video_scores(scores, records)
    mask = ad_mask(scores, info["threshold"], smooth, min_ad, max_gap)
    ads = ad_time_intervals(times, mask, duration, sorted(set(state_boundaries)))
    pieces = split_ad_intervals(intervals, ads, min_interval_sec)
    guard = ad_guard(ads, duration, max_runs, max_share) if mode == "flag" else None

    result = []
    for start, end, source in pieces:
        number = len(result)
        if source is None:
            # representative frame: the 1 Hz sample nearest the middle of the ad
            middle = (start + end) / 2
            sample = min(records, key=lambda item: abs(item["time_sec"] - middle))
            name = f"keyframes/ad_{number:05d}.jpg"
            shutil.copyfile(Path(directory) / sample["image_path"], Path(directory) / name)
            result.append({
                "start_sec": round(start, 6), "end_sec": round(end, 6),
                "keyframe_time_sec": round(sample["time_sec"], 6), "image_path": name,
                "content_kind": "ad" if mode == "cut" else "unclassified",
                "sharpness_score": round(sample["sharpness"], 3),
                "boundary_method": "ad_filter", "reveal_states": [],
                **({"ad_flagged": guard is None} if mode == "flag" else {}),
            })
            continue
        item = dict(intervals[source])
        item["start_sec"], item["end_sec"] = round(start, 6), round(end, 6)
        if not start <= item["keyframe_time_sec"] < end:
            # the slide's own keyframe lies in the ad: use its latest state outside the ad
            inside = [state for state in item["reveal_states"]
                      if start <= state["keyframe_time_sec"] < end]
            if inside:
                state = inside[-1]
                item["keyframe_time_sec"], item["image_path"] = state["keyframe_time_sec"], state["image_path"]
            else:
                sample = min((r for r in records if start <= r["time_sec"] < end),
                             key=lambda r: -r["sharpness"], default=None)
                if sample is not None:
                    name = f"keyframes/trim_{number:05d}.jpg"
                    shutil.copyfile(Path(directory) / sample["image_path"], Path(directory) / name)
                    item["keyframe_time_sec"], item["image_path"] = sample["time_sec"], name
            item["reveal_states"] = [state for state in item["reveal_states"]
                                     if state["end_sec"] > start and state["start_sec"] < end]
        if (start, end) != (intervals[source]["start_sec"], intervals[source]["end_sec"]):
            item["boundary_method"] = item["boundary_method"] + "+ad_trim"
        if mode == "flag":
            item["ad_flagged"] = False
        result.append(item)
    for number, item in enumerate(result):
        if mode == "flag":
            inside = [value for t, value in zip(times, scores) if item["start_sec"] <= t < item["end_sec"]]
            item["ad_score"] = round(sum(inside) / len(inside), 4) if inside else None
        item["segment_id"], item["sequence"] = f"{video_id}:seg_{number:05d}", number
        item["image_sha256"] = file_sha256(Path(directory) / item["image_path"])
    report = {**info, "mode": mode, "smooth": smooth, "min_ad_sec": min_ad, "max_gap_sec": max_gap,
              **({"guard": {"max_runs": max_runs, "max_share": max_share, "triggered": guard is not None,
                            "reason": guard}} if mode == "flag" else {}),
              "ad_intervals": [[round(a, 3), round(b, 3)] for a, b in ads],
              "ad_seconds": round(sum(b - a for a, b in ads), 1),
              "sample_times": [round(value, 3) for value in times],
              "sample_scores": [round(value, 4) for value in scores]}
    return result, report


## Video


In [ ]:
from pathlib import Path
VIDEOS = {
    'l25': ['L25_V003', 'L25_V004', 'L25_V006', 'L25_V008', 'L25_V009', 'L25_V010'],
    'ad_free': ['MLS-15DA1', 'MLS-03E08', 'MLS-2A8E1', 'MLS-C00E5',          # M3AV · Oxford
                'IIfeX44RI9k', 'dC93OcrNX6c', 'Xee8gmppUIw', '7EsHL8Sacg0',  # LPM · anat-1, bio-1, bio-4, dental
                'cs_0001', 'history_0000', 'agriculture_0002', 'music_0000'],  # SlideSpeech
    'vn_new': ['tHE1mC_n0-E', 'D_kuulxKuxc', 'bstCZZUMJYw', '_eDGTHhyQ7M', '8WnDooAH2a4', 'szRBlpJ0kkY',
               'laT0CKatosY', 'xysJI15jpAw', 'v2YTz8ToEag', 'hVU5Mw19V_c', 'xLGGWMDJRkY', 'y1CvB950VlU'],
}


## Trích tín hiệu


In [ ]:
import json, time, numpy as np, av, traceback
from pathlib import Path
from PIL import Image

OUT = Path('/kaggle/working/signals'); OUT.mkdir(parents=True, exist_ok=True)
videos = {p.stem: p for p in Path('/kaggle/input').rglob('*.mp4')}
student_path = next(iter(sorted(Path('/kaggle/input').rglob('ad_student.npz'))))
student = dict(np.load(student_path))
embed_images, embed_texts, scale, bias = load_siglip(str(student['backbone']), 'cuda:0')
gate = zero_shot(embed_images, embed_texts, scale, bias, CONTENT_PROMPTS, NO_CONTENT_PROMPTS)
gate_text = embed_texts(list(CONTENT_PROMPTS) + list(NO_CONTENT_PROMPTS))

def frames_1hz(path):
    next_time = 0.0
    with av.open(str(path)) as container:
        stream = container.streams.video[0]; stream.thread_type = 'AUTO'
        for frame in container.decode(stream):
            if frame.pts is None:
                continue
            t = float(frame.pts * frame.time_base)
            if t + 1e-6 < next_time:
                continue
            yield t, frame.to_image()
            next_time = t + 1.0

rows = []
for group, names in VIDEOS.items():
    for name in names:
        if name not in videos:
            print('THIẾU', name); continue
        started = time.monotonic()
        try:
            times, vectors, changes, batch, previous = [], [], [], [], None
            def flush():
                if batch:
                    vectors.append(embed_images([image for _, image in batch]))
                    batch.clear()
            for t, image in frames_1hz(videos[name]):
                signature = frame_signature(image)
                changes.append(0.0 if previous is None else float(signature_distance(previous, signature)))
                previous = signature
                times.append(t); batch.append((t, image.convert('RGB')))
                if len(batch) == 64:
                    flush()
            flush()
            emb = np.vstack(vectors).astype('float32')
            ad = 1 / (1 + np.exp(-(emb @ student['coef'] + float(student['intercept'][0]))))
            logits = emb @ gate_text.T * scale + bias
            logits -= logits.max(1, keepdims=True)
            probs = np.exp(logits) / np.exp(logits).sum(1, keepdims=True)
            content = probs[:, :len(CONTENT_PROMPTS)].sum(1)
            np.savez_compressed(OUT / f'{name}.npz', times=np.array(times, 'float32'), emb=emb.astype('float16'),
                                student=ad.astype('float32'), gate=content.astype('float32'),
                                change=np.array(changes, 'float32'))
            rows.append({'group': group, 'video': name, 'seconds': len(times), 'took': round(time.monotonic() - started)})
        except Exception:
            traceback.print_exc(); rows.append({'group': group, 'video': name, 'error': True})
        print(rows[-1], flush=True)
json.dump({'videos': VIDEOS, 'rows': rows}, open(OUT / 'index.json', 'w'), indent=1)
import shutil; shutil.make_archive('/kaggle/working/signals', 'zip', OUT)
